# SmritiMeds prompt eval and schedule render

Run the extraction prompt on one or more medication images and render the normalized schedule in a quick table.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from anthropic_client import analyze_medication_images
from config import load_config, load_env_file

load_env_file(PROJECT_ROOT / '.env')
config = load_config()
assert config.api_key, 'Set ANTHROPIC_API_KEY in environment or .env first.'
image_dir = Path(os.environ.get('SMRITIMEDS_EVAL_DIR', PROJECT_ROOT / 'sample_images'))
image_dir.mkdir(exist_ok=True)
images = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}])
print(f'Found {len(images)} image(s) in {image_dir}')


In [ ]:
def media_type(path: Path) -> str:
    suffix = path.suffix.lower()
    return {'.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg', '.webp': 'image/webp'}.get(suffix, 'image/jpeg')

results = []
for path in images:
    parsed, raw_json = analyze_medication_images(
        config,
        label_image_bytes=path.read_bytes(),
        label_mime_type=media_type(path),
    )
    results.append({'file': path.name, 'parsed': parsed, 'raw_json': raw_json})

results


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

rows = []
for result in results:
    parsed = result['parsed']
    if not parsed['schedule']:
        rows.append({
            'file': result['file'],
            'medication_name': parsed['medication_name'],
            'time_of_day': None,
            'label': None,
            'dose': None,
            'needs_manual_review': parsed['needs_manual_review'],
        })
        continue
    for entry in parsed['schedule']:
        rows.append({
            'file': result['file'],
            'medication_name': parsed['medication_name'],
            'time_of_day': entry['time_of_day'],
            'label': entry['label'],
            'dose': entry['dose'],
            'needs_manual_review': parsed['needs_manual_review'],
        })

if pd is None:
    rows
else:
    pd.DataFrame(rows)
